# Ribo-seq / RNA-seq Integrated Analysis

**Dataset:** GEO Series - Human tumor and normal ribosome profiling + RNA-seq  
**Reference:** GRCh38.p14 (GENCODE v46)

| SRA ID | GEO ID | Sample | Type |
|--------|--------|--------|------|
| SRR1562541 | - | Unknown A | riboseq |
| SRR1562542 | - | Unknown B | riboseq |
| SRR1562543 | GSM1495248 | Tumor Ribo-seq B | riboseq |
| SRR1562544 | GSM1495249 | Normal RNA-seq A | rnaseq |
| SRR1562545 | GSM1495250 | Normal RNA-seq B | rnaseq |
| SRR1562546 | GSM1495251 | Normal RNA-seq C | rnaseq |
| SRR1562547 | GSM1495252 | Tumor RNA-seq A | rnaseq |
| SRR1562548 | GSM1495253 | Tumor RNA-seq B | rnaseq |

## Environment Setup
```bash
conda env create -f environment.yml
conda activate patrick_rnaseq
python -m ipykernel install --user --name patrick_rnaseq --display-name "Patrick RNA-seq"
```
Then select the **Patrick RNA-seq** kernel.

## 1. Setup and Configuration

In [ ]:
import json, re, subprocess, gzip
from pathlib import Path
import numpy as np
import pandas as pd
import pysam
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

BASE = Path("/Volumes/Felix_SSD/Patrick")
REF_DIR, TRIM_DIR, ALIGN_DIR, BIGWIG_DIR = (BASE/d for d in ["reference","trimmed","aligned","bigwig"])
STAR_INDEX = REF_DIR / "STAR_index"
GENOME_FA = REF_DIR / "GRCh38.primary_assembly.genome.fa"
GTF = REF_DIR / "gencode.v46.annotation.gtf"
for d in [REF_DIR, TRIM_DIR, ALIGN_DIR, BIGWIG_DIR]: d.mkdir(exist_ok=True)

SAMPLES = {
    "SRR1562541": {"name": "Unknown Ribo-seq A", "type": "riboseq"},
    "SRR1562542": {"name": "Unknown Ribo-seq B", "type": "riboseq"},
    "SRR1562543": {"name": "Tumor Ribo-seq B",   "type": "riboseq"},
    "SRR1562544": {"name": "Normal RNA-seq A",   "type": "rnaseq"},
    "SRR1562545": {"name": "Normal RNA-seq B",   "type": "rnaseq"},
    "SRR1562546": {"name": "Normal RNA-seq C",   "type": "rnaseq"},
    "SRR1562547": {"name": "Tumor RNA-seq A",    "type": "rnaseq"},
    "SRR1562548": {"name": "Tumor RNA-seq B",    "type": "rnaseq"},
}

def find_fastq(sra_id):
    for p in BASE.glob("*.fastq"):
        if sra_id in p.name: return p
    return None

# Okabe-Ito colorblind-safe palette (scientific-visualization skill)
OKABE_ITO = ["#E69F00","#56B4E9","#009E73","#F0E442","#0072B2","#D55E00","#CC79A7","#000000"]
SAMPLE_COLORS = {sra: OKABE_ITO[i] for i, sra in enumerate(SAMPLES)}
TYPE_COLORS = {"riboseq": "#56B4E9", "rnaseq": "#E69F00"}

def apply_publication_style(fig, width=1000, height=600, x_grid=False, y_grid=True, font_size=13):
    \"\"\"Nature/Science pub styling (plotly skill template).\"\"\"
    fig.update_layout(plot_bgcolor="white", paper_bgcolor="white", width=width, height=height,
        font=dict(family="Arial, Helvetica, sans-serif", size=font_size, color="black"),
        margin=dict(l=80, r=40, t=60, b=60))
    ax = dict(showline=True, linewidth=2, linecolor="black", tickcolor="black", ticks="outside")
    fig.update_xaxes(**ax, showgrid=x_grid, gridcolor="lightgray")
    fig.update_yaxes(**ax, showgrid=y_grid, gridcolor="lightgray")
    return fig

def set_legend_position(fig, position="top-right"):
    \"\"\"Legend positioning helper (plotly skill).\"\"\"
    p = {"top-left":(0.99,0.01,"top","left"),"top-right":(0.99,0.99,"top","right"),
         "bottom-left":(0.01,0.01,"bottom","left"),"bottom-right":(0.01,0.99,"bottom","right")}
    y,x,ya,xa = p.get(position, p["top-right"])
    fig.update_layout(legend=dict(yanchor=ya,y=y,xanchor=xa,x=x,bgcolor="rgba(255,255,255,0.8)",borderwidth=0))
    return fig

for sra, m in SAMPLES.items():
    fq = find_fastq(sra)
    print(f"  {sra} [{m['type']:7s}] {m['name']:25s} {'FOUND' if fq else 'MISSING'}")

## 2. Reference Genome Download and Indexing

In [ ]:
GENCODE = "https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_46"
for local, url in {GENOME_FA.with_suffix(".fa.gz"): f"{GENCODE}/GRCh38.primary_assembly.genome.fa.gz",
                   GTF.with_suffix(".gtf.gz"): f"{GENCODE}/gencode.v46.annotation.gtf.gz"}.items():
    if local.exists() or local.with_suffix("").exists(): print(f"  Exists: {local.name}"); continue
    print(f"  Downloading {local.name}...")
    subprocess.run(["curl","-L","-o",str(local),url], check=True)
for gz in [GENOME_FA.with_suffix(".fa.gz"), GTF.with_suffix(".gtf.gz")]:
    dec = gz.with_suffix("")
    if not dec.exists() and gz.exists():
        print(f"  Decompressing {gz.name}..."); subprocess.run(["gunzip","-k",str(gz)], check=True)
    else: print(f"  Ready: {dec.name}")

In [ ]:
STAR_INDEX.mkdir(exist_ok=True)
if (STAR_INDEX / "SA").exists(): print("STAR index exists.")
else:
    print("Building STAR index (~30 min)...")
    r = subprocess.run(["STAR","--runMode","genomeGenerate","--genomeDir",str(STAR_INDEX),
        "--genomeFastaFiles",str(GENOME_FA),"--sjdbGTFfile",str(GTF),
        "--runThreadN","8","--sjdbOverhang","100"], capture_output=True, text=True)
    print("Done." if r.returncode==0 else f"ERROR: {r.stderr[-500:]}")

## 3. Adapter Trimming (fastp)
Ribo-seq: 20-35 nt length filter | RNA-seq: 20 nt minimum

In [ ]:
def run_fastp(sra_id, stype):
    out = TRIM_DIR / f"{sra_id}_trimmed.fastq.gz"
    if out.exists(): print(f"  {sra_id}: exists"); return
    fq = find_fastq(sra_id)
    if not fq: print(f"  {sra_id}: not found"); return
    cmd = ["fastp","-i",str(fq),"-o",str(out),"--html",str(TRIM_DIR/f"{sra_id}_fastp.html"),
           "--json",str(TRIM_DIR/f"{sra_id}_fastp.json"),"--thread","8","--length_required","20"]
    if stype=="riboseq": cmd+=["--length_limit","35"]
    print(f"  {sra_id}: trimming...")
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(f"  {sra_id}: {'done' if r.returncode==0 else 'ERROR'}")

for s,m in SAMPLES.items(): run_fastp(s, m["type"])

## 4. QC Summary

In [ ]:
qc_rows = []
for sra_id, meta in SAMPLES.items():
    jp = TRIM_DIR / f"{sra_id}_fastp.json"
    if not jp.exists(): continue
    rpt = json.loads(jp.read_text())
    b, a, f = rpt["summary"]["before_filtering"], rpt["summary"]["after_filtering"], rpt["filtering_result"]
    qc_rows.append({"Sample": meta["name"], "SRA": sra_id, "Type": meta["type"],
        "Raw reads (M)": b["total_reads"]/1e6, "Passed reads (M)": a["total_reads"]/1e6,
        "Pass rate (%)": f["passed_filter_reads"]/b["total_reads"]*100,
        "Adapter trimmed (M)": rpt.get("adapter_cutting",{}).get("adapter_trimmed_reads",0)/1e6,
        "Duplication (%)": rpt.get("duplication",{}).get("rate",0)*100,
        "GC (%)": b["gc_content"]*100})
qc_df = pd.DataFrame(qc_rows)
qc_df

In [ ]:
fig = make_subplots(rows=1, cols=3, subplot_titles=("Read counts","Duplication rate","GC content"), horizontal_spacing=0.08)
colors = [SAMPLE_COLORS[s] for s in qc_df["SRA"]]
fig.add_trace(go.Bar(name="Raw", x=qc_df["Sample"], y=qc_df["Raw reads (M)"], marker_color="lightgray"), row=1, col=1)
fig.add_trace(go.Bar(name="Passed", x=qc_df["Sample"], y=qc_df["Passed reads (M)"], marker_color=colors), row=1, col=1)
fig.add_trace(go.Bar(x=qc_df["Sample"], y=qc_df["Duplication (%)"], marker_color=colors, showlegend=False), row=1, col=2)
fig.add_trace(go.Bar(x=qc_df["Sample"], y=qc_df["GC (%)"], marker_color=colors, showlegend=False), row=1, col=3)
fig.update_layout(barmode="group")
fig.update_yaxes(title_text="Reads (millions)", row=1, col=1)
fig.update_yaxes(title_text="Duplication (%)", row=1, col=2)
fig.update_yaxes(title_text="GC (%)", row=1, col=3)
fig.update_xaxes(tickangle=-40)
apply_publication_style(fig, width=1400, height=500); set_legend_position(fig, "top-left")
fig.show()

## 5. STAR Alignment
Run **sequentially** (STAR loads ~30 GB genome index into memory).

In [ ]:
def run_star(sra_id, stype):
    bam = ALIGN_DIR / f"{sra_id}_Aligned.sortedByCoord.out.bam"
    if bam.exists() and bam.stat().st_size > 0:
        print(f"  {sra_id}: BAM exists ({bam.stat().st_size/1e9:.1f} GB)"); return
    fq = TRIM_DIR / f"{sra_id}_trimmed.fastq.gz"
    if not fq.exists(): print(f"  {sra_id}: no trimmed FASTQ"); return
    cmd = ["STAR","--runThreadN","8","--genomeDir",str(STAR_INDEX),"--readFilesIn",str(fq),
           "--readFilesCommand","zcat","--outSAMtype","BAM","SortedByCoordinate",
           "--outFileNamePrefix",str(ALIGN_DIR/f"{sra_id}_"),"--quantMode","GeneCounts",
           "--outSAMattributes","NH","HI","AS","NM","MD"]
    if stype=="riboseq": cmd+=["--alignIntronMax","1","--outFilterMismatchNmax","2","--outFilterMultimapNmax","1"]
    print(f"  {sra_id}: aligning ({stype})...")
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode!=0: print(f"  ERROR: {r.stderr[-500:]}"); return
    subprocess.run(["samtools","index",str(bam)])
    print(f"  {sra_id}: done ({bam.stat().st_size/1e9:.1f} GB)")

for s,m in SAMPLES.items(): run_star(s, m["type"])

In [ ]:
def parse_star_log(sid):
    lp = ALIGN_DIR / f"{sid}_Log.final.out"
    if not lp.exists(): return None
    stats = {}
    for line in lp.read_text().splitlines():
        if "|" in line:
            k, v = line.split("|",1); k, v = k.strip(), v.strip().rstrip("%")
            try: stats[k] = float(v)
            except ValueError: stats[k] = v
    return stats

align_rows = []
for sid, meta in SAMPLES.items():
    s = parse_star_log(sid)
    if not s: continue
    align_rows.append({"Sample": meta["name"], "SRA": sid, "Type": meta["type"],
        "Input reads (M)": s.get("Number of input reads",0)/1e6,
        "Uniquely mapped (%)": s.get("Uniquely mapped reads %",0),
        "Multi-mapped (%)": s.get("% of reads mapped to multiple loci",0),
        "Unmapped too short (%)": s.get("% of reads unmapped: too short",0)})
align_df = pd.DataFrame(align_rows)

fig = go.Figure()
for col, color in [("Uniquely mapped (%)","#009E73"),("Multi-mapped (%)","#E69F00"),("Unmapped too short (%)","#D55E00")]:
    fig.add_trace(go.Bar(name=col.replace(" (%)",""), x=align_df["Sample"], y=align_df[col], marker_color=color))
fig.update_layout(barmode="stack", title="STAR alignment rates", yaxis_title="Reads (%)", yaxis_range=[0,105], xaxis_tickangle=-30)
apply_publication_style(fig); set_legend_position(fig, "top-right")
fig.show()
align_df

## 6. Coverage Tracks (deepTools bamCoverage)
CPM-normalized bigWig for downstream `computeMatrix`.

In [ ]:
for sid in SAMPLES:
    bam = ALIGN_DIR / f"{sid}_Aligned.sortedByCoord.out.bam"
    bw = BIGWIG_DIR / f"{sid}.bw"
    if bw.exists(): print(f"  {sid}: exists"); continue
    if not bam.exists() or bam.stat().st_size==0: print(f"  {sid}: no BAM"); continue
    print(f"  {sid}: bamCoverage...")
    r = subprocess.run(["bamCoverage","--bam",str(bam),"--outFileName",str(bw),
        "--normalizeUsing","CPM","--binSize","10","--numberOfProcessors","8"],
        capture_output=True, text=True)
    print(f"  {sid}: {'done' if r.returncode==0 else 'ERROR'}")

## 7. Gene-Level Counting (featureCounts)

In [ ]:
counts_file = ALIGN_DIR / "gene_counts.txt"
if not counts_file.exists():
    bams = [str(ALIGN_DIR/f"{s}_Aligned.sortedByCoord.out.bam") for s in SAMPLES
            if (ALIGN_DIR/f"{s}_Aligned.sortedByCoord.out.bam").exists()
            and (ALIGN_DIR/f"{s}_Aligned.sortedByCoord.out.bam").stat().st_size>0]
    print(f"featureCounts on {len(bams)} BAMs...")
    r = subprocess.run(["featureCounts","-a",str(GTF),"-o",str(counts_file),"-T","8",
        "-t","exon","-g","gene_id","--primary"]+bams, capture_output=True, text=True)
    print("Done." if r.returncode==0 else f"ERROR: {r.stderr[-500:]}")
else: print("Exists.")

In [ ]:
raw = pd.read_csv(counts_file, sep="\t", comment="#")
col_map = {c: s for c in raw.columns for s in SAMPLES if s in c}
raw = raw.rename(columns=col_map)
raw["gene_id"] = raw["Geneid"].str.split(".").str[0]
available_sra = [s for s in SAMPLES if s in raw.columns]
counts = raw.set_index("gene_id")[available_sra].copy()
counts.columns = [SAMPLES[s]["name"] for s in available_sra]
counts = counts.loc[counts.sum(axis=1) > 0]
print(f"Genes with >0 reads: {len(counts):,}")
counts.head(10)

In [ ]:
fig = go.Figure()
for i, col in enumerate(counts.columns):
    lc = np.log10(counts[col].replace(0,np.nan).dropna())
    fig.add_trace(go.Histogram(x=lc, name=col, opacity=0.5, nbinsx=80, marker_color=OKABE_ITO[i%len(OKABE_ITO)]))
fig.update_layout(title="Gene read count distribution", xaxis_title="log10(read count)", yaxis_title="Number of genes", barmode="overlay")
apply_publication_style(fig); set_legend_position(fig, "top-right")
fig.show()

In [ ]:
riboseq_cols = [SAMPLES[s]["name"] for s in available_sra if SAMPLES[s]["type"]=="riboseq"]
rnaseq_cols  = [SAMPLES[s]["name"] for s in available_sra if SAMPLES[s]["type"]=="rnaseq"]
counts["Ribo-seq avg"] = counts[riboseq_cols].mean(axis=1)
counts["RNA-seq avg"]  = counts[rnaseq_cols].mean(axis=1)

top = counts.nlargest(30, "Ribo-seq avg")
fig = go.Figure()
fig.add_trace(go.Bar(name="Ribo-seq", y=top.index, x=top["Ribo-seq avg"], orientation="h", marker_color=TYPE_COLORS["riboseq"]))
fig.add_trace(go.Bar(name="RNA-seq",  y=top.index, x=top["RNA-seq avg"],  orientation="h", marker_color=TYPE_COLORS["rnaseq"]))
fig.update_layout(title="Top 30 genes by Ribo-seq read count", xaxis_title="Average read count", barmode="group")
apply_publication_style(fig, height=800); set_legend_position(fig, "bottom-right")
fig.show()

## 8. Translational Efficiency
$$TE = \\frac{\\text{Ribo-seq RPM}}{\\text{RNA-seq RPM}}$$

In [ ]:
ribo_rpm = counts["Ribo-seq avg"]/counts["Ribo-seq avg"].sum()*1e6
rnaseq_rpm = counts["RNA-seq avg"]/counts["RNA-seq avg"].sum()*1e6
mask = (counts["Ribo-seq avg"]>=10) & (counts["RNA-seq avg"]>=10)
te_df = pd.DataFrame({"ribo_rpm": ribo_rpm[mask], "rnaseq_rpm": rnaseq_rpm[mask],
    "ribo_counts": counts["Ribo-seq avg"][mask], "rnaseq_counts": counts["RNA-seq avg"][mask]})
te_df["TE"] = te_df["ribo_rpm"]/te_df["rnaseq_rpm"]
te_df["log2_TE"] = np.log2(te_df["TE"])
te_df = te_df.replace([np.inf,-np.inf], np.nan).dropna()
print(f"Genes: {len(te_df):,} | Median TE: {te_df['TE'].median():.2f}")
print(f"High TE (>2): {(te_df['TE']>2).sum():,} | Low TE (<0.5): {(te_df['TE']<0.5).sum():,}")

In [ ]:
# TE scatter (PuOr diverging - colorblind-safe, scientific-visualization skill)
fig = px.scatter(te_df.reset_index(), x=np.log10(te_df["rnaseq_rpm"]), y=np.log10(te_df["ribo_rpm"]),
    color="log2_TE", color_continuous_scale="PuOr_r", color_continuous_midpoint=0,
    hover_data={"gene_id": te_df.index},
    labels={"x":"log10(RNA-seq RPM)","y":"log10(Ribo-seq RPM)","color":"log2(TE)"},
    title="Translational Efficiency")
rng = [np.log10(max(te_df[["rnaseq_rpm","ribo_rpm"]].min().min(),0.01)),
       np.log10(te_df[["rnaseq_rpm","ribo_rpm"]].max().max())]
fig.add_trace(go.Scatter(x=rng, y=rng, mode="lines", line=dict(dash="dash",color="gray",width=1), name="TE = 1"))
fig.update_traces(marker=dict(size=4, opacity=0.6), selector=dict(mode="markers"))
apply_publication_style(fig, width=800, height=700)
fig.show()

In [ ]:
te_sorted = te_df.sort_values("TE")
extreme = pd.concat([te_sorted.head(30), te_sorted.tail(30)])
fig = go.Figure(go.Bar(y=extreme.index, x=extreme["log2_TE"], orientation="h",
    marker_color=["#D55E00" if v<0 else "#009E73" for v in extreme["log2_TE"]]))
fig.add_vline(x=0, line_dash="dash", line_color="gray")
fig.update_layout(title="Extreme TE genes (top/bottom 30)", xaxis_title="log2(TE)")
apply_publication_style(fig, height=900)
fig.show()

## 9. 3'UTR Coverage Heatmaps (deepTools + Plotly)

deepTools `computeMatrix scale-regions` computes normalized coverage across 3'UTR
regions from bigWig tracks (orders of magnitude faster than pysam pileup).
Heatmaps use perceptually uniform colormaps (`Inferno`, `Viridis`) - colorblind-safe
and grayscale-compatible (per scientific-visualization skill).

In [ ]:
utr_bed = BASE / "three_prime_utrs.bed"
if not utr_bed.exists():
    recs = []
    for line in open(GTF):
        if line.startswith("#"): continue
        f = line.strip().split("\t")
        if f[2]!="three_prime_utr": continue
        gid = re.search(r'gene_id "([^"]+)"', f[8])
        gn = re.search(r'gene_name "([^"]+)"', f[8])
        if gid:
            gi = gid.group(1).split(".")[0]
            recs.append((f[0], int(f[3])-1, int(f[4]), gn.group(1) if gn else gi, int(f[4])-int(f[3])+1, f[6], gi))
    udf = pd.DataFrame(recs, columns=["chrom","start","end","name","length","strand","gene_id"])
    udf = udf.sort_values("length",ascending=False).drop_duplicates("gene_id",keep="first").query("length>=200")
    udf = udf[udf["gene_id"].isin(te_df.index)]
    udf = udf.merge(te_df[["ribo_counts"]].rename(columns={"ribo_counts":"score"}), left_on="gene_id", right_index=True)
    udf = udf.nlargest(500, "score")
    udf[["chrom","start","end","name","score","strand"]].to_csv(utr_bed, sep="\t", header=False, index=False)
    print(f"Wrote {len(udf)} 3'UTR regions")
else: print("BED exists")

In [ ]:
matrix_gz = BASE / "utr_coverage_matrix.gz"
ribo_bws = [str(BIGWIG_DIR/f"{s}.bw") for s in SAMPLES if SAMPLES[s]["type"]=="riboseq" and (BIGWIG_DIR/f"{s}.bw").exists()]
rnaseq_bws = [str(BIGWIG_DIR/f"{s}.bw") for s in SAMPLES if SAMPLES[s]["type"]=="rnaseq" and (BIGWIG_DIR/f"{s}.bw").exists()]

if not matrix_gz.exists() and ribo_bws and rnaseq_bws:
    bws = [ribo_bws[0], rnaseq_bws[0]]
    print(f"Ribo: {Path(bws[0]).name} | RNA: {Path(bws[1]).name}")
    r = subprocess.run(["computeMatrix","scale-regions","-S"]+bws+["-R",str(utr_bed),
        "--regionBodyLength","1000","--beforeRegionStartLength","200","--afterRegionStartLength","200",
        "-o",str(matrix_gz),"--numberOfProcessors","8","--skipZeros"], capture_output=True, text=True)
    print("Done." if r.returncode==0 else f"ERROR: {r.stderr[-500:]}")
else: print("Matrix exists or bigWigs unavailable.")

In [ ]:
with gzip.open(matrix_gz, "rt") as fh:
    hdr = json.loads(fh.readline().lstrip("@"))
    md = pd.read_csv(fh, sep="\t", header=None)
gene_names = md.iloc[:,3].values
sb = hdr.get("sample_boundaries",[])
if len(sb)>=3:
    bps = sb[1]-sb[0]
    ribo_mat = md.iloc[:,6:6+bps].values.astype(float)
    rnaseq_mat = md.iloc[:,6+bps:6+2*bps].values.astype(float)
else:
    h = (md.shape[1]-6)//2
    ribo_mat = md.iloc[:,6:6+h].values.astype(float)
    rnaseq_mat = md.iloc[:,6+h:].values.astype(float)
ribo_mat, rnaseq_mat = np.nan_to_num(ribo_mat), np.nan_to_num(rnaseq_mat)
print(f"Matrix: {ribo_mat.shape[0]} genes x {ribo_mat.shape[1]} bins")

In [ ]:
def row_norm(m):
    mx = m.max(axis=1, keepdims=True); mx[mx==0]=1; return m/mx
ribo_n, rna_n = row_norm(ribo_mat), row_norm(rnaseq_mat)
nc = ribo_mat.shape[1]
f3, l3 = ribo_mat[:,:nc//3].mean(1), ribo_mat[:,2*nc//3:].mean(1)
f3[f3==0]=0.01; rt = l3/f3
has = ribo_mat.sum(1)>0; si = np.argsort(rt[has])[::-1]
rd, rnd, nd = ribo_n[has][si], rna_n[has][si], gene_names[has][si]
print(f"Genes with coverage: {has.sum()}")

In [ ]:
n = min(200, len(rd))
fig = make_subplots(rows=1, cols=2, subplot_titles=("Ribo-seq 3'UTR","RNA-seq 3'UTR"),
    shared_yaxes=True, horizontal_spacing=0.05)
fig.add_trace(go.Heatmap(z=rd[:n], y=nd[:n], colorscale="Inferno",
    colorbar=dict(title="Norm.", x=0.46, len=0.8)), row=1, col=1)
fig.add_trace(go.Heatmap(z=rnd[:n], y=nd[:n], colorscale="Viridis",
    colorbar=dict(title="Norm.", x=1.02, len=0.8)), row=1, col=2)
fig.update_layout(title="3'UTR coverage (sorted by read-through score)",
    height=max(600,n*4), width=1200, plot_bgcolor="white", paper_bgcolor="white",
    font=dict(family="Arial", size=11, color="black"))
fig.update_xaxes(title_text="Scaled 3'UTR position (5'->3')")
fig.update_yaxes(title_text="Gene", row=1, col=1, autorange="reversed")
fig.show()

### Read-through Candidates
Genes at top show sustained Ribo-seq signal into the 3'UTR (potential read-through).

In [ ]:
rtdf = pd.DataFrame({"gene_name": nd, "readthrough_score": rt[has][si],
    "ribo_3utr_mean": ribo_mat[has][si].mean(1), "rnaseq_3utr_mean": rnaseq_mat[has][si].mean(1)})
rtdf = rtdf[rtdf["ribo_3utr_mean"]>1]
print("Read-through candidates (top 20):")
rtdf.head(20)